# Corrected Sigma_0: Adam, SDE, and ODE

This demo compares streaming Adam with the order-zero Sigma_0 SDE and ODE. The experiment is seeded and uses diagonal data covariance by default, so it is suitable for CPU or GPU execution.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import config
config.enable_x64()
import dynamics
import problems
import simulate

print('backend:', jax.default_backend())
print('devices:', jax.devices())
print('x64:', jax.config.jax_enable_x64)

: 

In [ ]:
# Keep these values modest for an interactive GPU demo.
d, m = 128, 1
beta1, beta2 = 0.1, 0.1
eps = 1e-3
lr = 0.7                    # continuous-time SDE/ODE learning rate
adam_lr = lr / d            # one Adam step advances time by 1 / d
T = 2.0
adam_steps = int(T * d)
dt_sde, dt_ode = 0.02, 0.05
mc_samples, sigma_samples, sigma_history = 5000, 1500, 30
sde_replicates = 16

key = jax.random.PRNGKey(123)
k_params, k_opt, k_run, k_sde, k_ode = jax.random.split(key, 5)
params0 = jax.random.normal(k_params, (d, m)) / jnp.sqrt(d)
optimal = jax.random.normal(k_opt, (d, m)) / jnp.sqrt(d)
optimal = optimal / jnp.linalg.norm(optimal, axis=0) * 5.0
cov = jnp.arange(1, d + 1, dtype=jnp.float64) ** -0.5
problem = problems.get_problem('linreg')
adam = simulate.build_adam(problem, cov, optimal, adam_lr, beta1=beta1, beta2=beta2, eps=eps)
_, adam_risks = simulate.run(adam, problem, params0, optimal, cov, adam_steps, key=k_run)
print('initial/final Adam risk:', float(adam_risks[0]), float(adam_risks[-1]))

initial/final Adam risk: 1.99093212824933 1.334394870746158


In [ ]:
ode_risks, ode_time = dynamics.run_adam_ode(
    problem, params0, optimal, cov, T, lr, beta1=beta1, beta2=beta2,
    dt=dt_ode, num_samples=mc_samples, eps=eps, key=k_ode,
)

sde_keys = jax.random.split(k_sde, sde_replicates)
def one_sde(k):
    return dynamics.run_adam_sde(
        problem, params0, optimal, cov, T, lr, beta1=beta1, beta2=beta2,
        dt=dt_sde, num_samples=mc_samples, eps=eps,
        noise_samples=sigma_samples, noise_history=sigma_history, key=k,
    )[0]
sde_risks = jax.vmap(one_sde)(sde_keys)
sde_time = jnp.arange(sde_risks.shape[1]) * dt_sde
sde_mean = sde_risks.mean(axis=0)
sde_lo, sde_hi = jnp.percentile(sde_risks, jnp.array([10., 90.]), axis=0)

plt.figure(figsize=(9, 5))
plt.plot(jnp.arange(adam_steps) / d, adam_risks, label='streaming Adam', lw=2)
plt.plot(ode_time, ode_risks, label='Sigma_0 ODE', lw=2)
plt.plot(sde_time, sde_mean, label='Sigma_0 SDE mean', lw=2)
plt.fill_between(sde_time, sde_lo, sde_hi, alpha=0.2, label='SDE 10-90%')
plt.xlabel('continuous time')
plt.ylabel('population risk')
plt.title('True Adam versus corrected Sigma_0 dynamics')
plt.grid(alpha=0.25)
plt.legend()
plt.show()

print('final ODE risk:', float(ode_risks[-1]))
print('final SDE mean risk:', float(sde_mean[-1]))
print('final SDE 10-90%:', float(sde_lo[-1]), float(sde_hi[-1]))

## Dense covariance sanity check

The dense path retains coordinate correlations through the normalized correlation matrix. This small check exercises the same corrected Sigma_0 sampler without the large default allocation.

In [ ]:
from discounts import sigma0_samples
dd = 12
small_cov = jnp.arange(1, dd + 1, dtype=jnp.float64) ** -0.5
qmat, _ = jnp.linalg.qr(jax.random.normal(jax.random.PRNGKey(77), (dd, dd)))
K = (qmat * small_cov) @ qmat.T
K = (K + K.T) / 2
corr = K / jnp.sqrt(jnp.diag(K))[:, None]
A0 = sigma0_samples(
    jnp.array([[2.0, 0.4], [0.4, 1.5]]), problem.f, beta1, beta2,
    jax.random.PRNGKey(78), jnp.linalg.cholesky(corr),
    num_samples=512, history_length=20, eps=eps,
)
print('dense A_0 samples:', A0.shape)
print('finite:', bool(jnp.all(jnp.isfinite(A0))))